# Iris Classification with Blind Insight Data

This notebook demonstrates how to use data from Blind Insight for machine learning tasks with scikit-learn.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the backend API is running:
   ```bash
   cd cube-server
   node index.js
   ```

3. Make sure you have uploaded the Iris dataset to Blind Insight using the importer tool.


## Setup and Imports


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient, load_iris_from_blind


## Load Data from Blind Insight

Instead of using `datasets.load_iris()`, we load the data from Blind Insight.

**Note**: Update the organization, dataset_slug, and schema_slug to match your Blind Insight setup.


### Workflows
- **Encrypted-only (recommended)**: Use Blind Insight encrypted primitives (count, avg, min, max, ranges) and keep rows encrypted. See the *Encrypted Aggregation Classifier* section below.
- **Plaintext ML (illustrative only)**: Decrypt data and run scikit-learn directly. This is not encrypted and is kept for reference/testing.


In [2]:
# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
DATASET_SLUG = "fraud-analysis-training"
SCHEMA_SLUG = "fraud-analysis-schema"
API_URL = "https://proxy.local.blindinsight.io/"

USERNAME = "data_owner@localhost"  # Your Blind Insight account email
PASSWORD = "blindinsight"  # Your Blind Insight account password

SCHEMA_ID = "iEbXomadeyp3JcDfZMFLYp"

# Plaintext load (decrypt=True inside load_iris_from_blind)
# This replaces: iris = datasets.load_iris()
# X, y = load_iris_from_blind(
#     organization=ORGANIZATION,
#     dataset_slug=DATASET_SLUG,
#     schema_slug=SCHEMA_SLUG,
#     api_url=API_URL,
#     decrypt=True  # plaintext for scikit-learn; not encrypted
# )

client = BlindInsightClient(
    api_url=API_URL,
    username=USERNAME,
    password=PASSWORD,
    verify_ssl=False
)

# Load full dataset as DataFrame
df_raw = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=10000,  # Load enough data to have 500 fraud cases
    decrypt=True,
    schema_id=SCHEMA_ID
)

print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")


Error: 

## Prepare Data for Classification

For this example, we'll use only the first two features and the first two classes (similar to the scikit-learn example).


In [14]:
# Use only the first two features and first 100 samples (first two classes)
# This matches the scikit-learn example: X = iris.data[:100, :2]
X_subset = X[:100, :2]  # First 100 samples, first 2 features
y_subset = y[:100]  # First 100 samples (first two classes)

print(f"Subset shape: {X_subset.shape}")
print(f"Target classes in subset: {np.unique(y_subset)}")


Subset shape: (100, 2)
Target classes in subset: [0 1]


In [15]:
print(X_subset)
print(y_subset)


[[51 35]
 [49 30]
 [47 32]
 [46 31]
 [50 36]
 [54 39]
 [46 34]
 [50 34]
 [44 29]
 [49 31]
 [54 37]
 [48 34]
 [48 30]
 [43 30]
 [58 40]
 [57 44]
 [54 39]
 [51 35]
 [57 38]
 [51 38]
 [54 34]
 [51 37]
 [46 36]
 [51 33]
 [48 34]
 [50 30]
 [50 34]
 [52 35]
 [52 34]
 [47 32]
 [48 31]
 [54 34]
 [52 41]
 [55 42]
 [49 31]
 [50 32]
 [55 35]
 [49 36]
 [44 30]
 [51 34]
 [50 35]
 [45 23]
 [44 32]
 [50 35]
 [51 38]
 [48 30]
 [51 38]
 [46 32]
 [53 37]
 [50 33]
 [70 32]
 [64 32]
 [69 31]
 [55 23]
 [65 28]
 [57 28]
 [63 33]
 [49 24]
 [66 29]
 [52 27]
 [50 20]
 [59 30]
 [60 22]
 [61 29]
 [56 29]
 [67 31]
 [56 30]
 [58 27]
 [62 22]
 [56 25]
 [59 32]
 [61 28]
 [63 25]
 [61 28]
 [64 29]
 [66 30]
 [68 28]
 [67 30]
 [60 29]
 [57 26]
 [55 24]
 [55 24]
 [58 27]
 [60 27]
 [54 30]
 [60 34]
 [67 31]
 [63 23]
 [56 30]
 [55 25]
 [55 26]
 [61 30]
 [58 26]
 [50 23]
 [56 27]
 [57 30]
 [57 29]
 [62 29]
 [51 25]
 [57 28]]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0

## Train Logistic Regression Model


In [16]:
# Create an instance of Logistic Regression Classifier and fit the data
logreg = LogisticRegression(C=1e5, max_iter=1000)
clf = logreg.fit(X_subset, y_subset)

print("Model trained successfully!")
print(clf)


Model trained successfully!
LogisticRegression(C=100000.0, max_iter=1000)


## Evaluate Model Accuracy


In [17]:
# Make predictions
y_pred = clf.predict(X_subset)

# Calculate accuracy
accuracy = accuracy_score(y_subset, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")


Accuracy: 1.0000 (100.00%)


Blind 

In [18]:
# Example: Use encrypted filters to narrow dataset before decryption
# Encrypted aggregation-based classifier (no decryption)
# Two-class demo: setosa vs versicolor using petal-length feature
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL)
feature = "petal-length"
class_a = "I. setosa"
class_b = "I. versicolor"

# Helper to extract aggregation value (supports both shapes)
# Shape A: {records: [{data: {value: X}}]}
# Shape B: {records: [{value: X, aggregation_type: ...}]}

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    # Shape A
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    # Shape B
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# NOTE: Blind Insight aggregation supports counts on encrypted numbers; to avoid
# unsupported floating-point avg on some deployments, we do a threshold search
# using encrypted counts only (no decryption).

# Candidate thresholds over petal-length range (Iris ~0 to ~7)
thresholds = np.linspace(0.0, 7.0, 71)  # step 0.1

# Precompute totals per class
count_a_total = agg_value(
    client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:count(0~1000)",
        extra_filters=[f"species:{class_a}"],
        decrypt=False
    )
)
count_b_total = agg_value(
    client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:count(0~1000)",
        extra_filters=[f"species:{class_b}"],
        decrypt=False
    )
)

# Thresholds over integer-scaled petal-length (e.g., 0..700 if 0..7 *10)
# Narrow search window and coarser step to speed up
thresholds = range(0, 201, 20)   # was 0..700 step 10

best_acc = -1
best_t = None
best_counts = None

for t in thresholds:
    count_a_left = agg_value(
        client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:count(<{t})",
            extra_filters=[f"species:{class_a}"],
            decrypt=False
        )
    )
    count_b_left = agg_value(
        client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:count(<{t})",
            extra_filters=[f"species:{class_b}"],
            decrypt=False
        )
    )

    correct = count_a_left + (count_b_total - count_b_left)
    total = count_a_total + count_b_total
    acc = correct / total if total else 0.0

    if acc > best_acc:
        best_acc = acc
        best_t = t
        best_counts = (count_a_left, count_b_left, correct, total)

print(f"Best threshold on encrypted counts: {best_t:.3f}")
print(f"Encrypted-rule accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Counts at best threshold -> class_a_left: {best_counts[0]:.0f}, class_b_left: {best_counts[1]:.0f}, correct: {best_counts[2]:.0f}/{best_counts[3]:.0f}")

print("\nRule: predict setosa if petal-length < threshold, else versicolor (encrypted counts only)")


Error: 

## Alternative: Using the Client Directly

You can also use the client directly for more control over the data loading process.


## Encrypted Aggregation Classifier (no decryption)

This section demonstrates classification using only Blind Insight's encrypted primitives:
- Use encrypted filters and aggregations (count/min/max/ranges) on encrypted data
- Never decrypt rows
- Build a simple rule-based classifier using aggregate stats only

**Dataset note**: For this demo we assume the numeric features are integer-scaled (e.g., original floats multiplied by 10) so that counts/ranges operate on integers. Adjust ranges accordingly if you change the scaling.

Approach (two-class demo: setosa vs versicolor):
1. Search thresholds over integer-scaled petal-length using encrypted `count(<t)` per class.
2. Pick the threshold with best accuracy on encrypted counts.
3. No row-level decryption; only aggregates are returned.


## Encrypted averages: sepal width per class
Use Blind Insight encrypted `avg` to compute the mean sepal width for each class (integer-scaled).


In [2]:
# Encrypted avg of sepal-width per class (integer-scaled data)
from blind_insight_client import BlindInsightClient

# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
DATASET_SLUG = "fraud-analysis-training"
SCHEMA_SLUG = "fraud-analysis-schema"
API_URL = "https://proxy.local.blindinsight.io/"

USERNAME = "data_owner@localhost"  # Your Blind Insight account email
PASSWORD = "blindinsight"  # Your Blind Insight account password

SCHEMA_ID = "i9yp3TutxNsCp3fgNWE4p2"

# Plaintext load (decrypt=True inside load_iris_from_blind)
# This replaces: iris = datasets.load_iris()
# X, y = load_iris_from_blind(
#     organization=ORGANIZATION,
#     dataset_slug=DATASET_SLUG,
#     schema_slug=SCHEMA_SLUG,
#     api_url=API_URL,
#     decrypt=True  # plaintext for scikit-learn; not encrypted
# )

# client = BlindInsightClient(
#     api_url=API_URL,
#     username=USERNAME,
#     password=PASSWORD,
#     verify_ssl=False
# )
#
# # Load full dataset as DataFrame
# df_raw = client.load_data(
#     organization=ORGANIZATION,
#     dataset_slug=DATASET_SLUG,
#     schema_slug=SCHEMA_SLUG,
#     limit=10000,  # Load enough data to have 500 fraud cases
#     decrypt=True,
#     schema_id=SCHEMA_ID
# )

#----------

client = BlindInsightClient(
    api_url=API_URL,
    username=USERNAME,
    password=PASSWORD,
    verify_ssl=False
)

# Load full dataset as DataFrame
# df_raw = client.load_data(
#     organization=ORGANIZATION,
#     dataset_slug=DATASET_SLUG,
#     schema_slug=SCHEMA_SLUG,
#     limit=10000,  # Load enough data to have 500 fraud cases
#     decrypt=True,
#     schema_id=SCHEMA_ID
# )

# client = BlindInsightClient(api_url=API_URL)
feature = "amount"
classes = [0, 1]

# Reuse agg_value from earlier cell if in scope; redefine defensively

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

for cls in classes:
    filter_str = f"is_fraud:{cls}"
    print(f"DEBUG: Using filter: {filter_str}")
    resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"{feature}:avg(0~1000)",
        extra_filters=[filter_str],
        decrypt=False,  # keep encrypted; avg is computed inside Blind Insight
        schema_id=SCHEMA_ID
    )
    print(f"DEBUG: Response keys: {resp.keys() if isinstance(resp, dict) else 'Not a dict'}")
    if isinstance(resp, dict) and 'records' in resp:
        print(f"DEBUG: Number of records: {len(resp['records'])}")
        if resp['records']:
            print(f"DEBUG: First record keys: {resp['records'][0].keys() if isinstance(resp['records'][0], dict) else 'Not a dict'}")
    mean_val = agg_value(resp)
    print(f"Mean {feature} for {cls}: {mean_val:.3f}")
    print()



DEBUG: Using filter: is_fraud:0
[DEBUG aggregate] Combined filters: ['is_fraud:0', 'amount:avg(0~1000)']
[DEBUG aggregate] extra_filters: ['is_fraud:0'], agg_filter: amount:avg(0~1000)
[DEBUG query] search_filters array: [{'label': 'is_fraud', 'value': '0'}, {'label': 'amount', 'value': 'avg(0~1000)'}]
[DEBUG query] Final payload JSON: {
  "schema": "i9yp3TutxNsCp3fgNWE4p2",
  "filters": [
    {
      "label": "is_fraud",
      "value": "0"
    },
    {
      "label": "amount",
      "value": "avg(0~1000)"
    }
  ],
  "limit": 1,
  "offset": 0
}


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "avg",
      "value": 9
    },
    "id": "agg-1768104439905569",
    "schema": "https://localhost:8080/api/schemas/gbY69UXqbtRJLvCJXevAAT/",
    "url": "https://localhost:8080/api/records/agg-1768104439905569/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'avg', 'value': 9}, 'id': 'agg-1768104439905569', 'schema': 'https://localhost:8080/api/schemas/gbY69UXqbtRJLvCJXevAAT/', 'url': 'https://localhost:8080/api/records/agg-1768104439905569/'}]}
DEBUG: Response keys: dict_keys(['success', 'records'])
DEBUG: Number of records: 1
DEBUG: First record keys: dict_keys(['data', 'id', 'schema', 'url'])
Mean amount for 0: 9.000

DEBUG: Using filter: is_fraud:1
[DEBUG aggregate] Combined filters: ['is_fraud:1', 'amount:avg(0~1000)']
[DEBUG aggregate] extra_filters: ['is_fraud:1'], agg_filter: amount:avg(0~1000)
[DEBUG query] search_filters array: [{'label': 'is_f

/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "avg",
      "value": 73
    },
    "id": "agg-1768104440334708",
    "schema": "https://localhost:8080/api/schemas/gbY69UXqbtRJLvCJXevAAT/",
    "url": "https://localhost:8080/api/records/agg-1768104440334708/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'avg', 'value': 73}, 'id': 'agg-1768104440334708', 'schema': 'https://localhost:8080/api/schemas/gbY69UXqbtRJLvCJXevAAT/', 'url': 'https://localhost:8080/api/records/agg-1768104440334708/'}]}
DEBUG: Response keys: dict_keys(['success', 'records'])
DEBUG: Number of records: 1
DEBUG: First record keys: dict_keys(['data', 'id', 'schema', 'url'])
Mean amount for 1: 73.000



In [6]:
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)

# Query for is_fraud:0
result_0 = client.query(
  organization=ORGANIZATION,
  dataset_slug=DATASET_SLUG,
  schema_slug=SCHEMA_SLUG,
  limit=10,
  filters=["is_fraud:0"],
  decrypt=True,  # Decrypt to see actual values
  schema_id=SCHEMA_ID
)

# Query for is_fraud:1
result_1 = client.query(
  organization=ORGANIZATION,
  dataset_slug=DATASET_SLUG,
  schema_slug=SCHEMA_SLUG,
  limit=10,
  filters=["is_fraud:1"],
  decrypt=True,  # Decrypt to see actual values
  schema_id=SCHEMA_ID
)

print(f"is_fraud:0 records found: {len(result_0['records'])}")
print(f"is_fraud:1 records found: {len(result_1['records'])}")

if result_0['records']:
  print(f"\nSample is_fraud:0 record: {result_0['records'][0]}")
if result_1['records']:
  print(f"\nSample is_fraud:1 record: {result_1['records'][0]}")


[DEBUG query] search_filters array: [{'label': 'is_fraud', 'value': '0'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp",
  "filters": [
    {
      "label": "is_fraud",
      "value": "0"
    }
  ],
  "limit": 10,
  "offset": 0
}


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "0461a05386c752640fb67219c1863c40b1774f930751aa180774b16c88d68d62": "d4c825400924747cda85960054ea116132632f39bf46877f3eb7dde2f206f008f748163f",
      "322db4f55171a681ce1b167427e3f53d2532c21960c2d19f479f9c86a8d8bc20": "28cd1508f036504e3a5ea03c41e967e127ef12f9de6c807a48cb92fd33b1cfe85ad1ff58",
      "3b9fbc9bf5f611b36303e1fa66fe9c10dc78104d499b7ff733ae7446cecda26c": "66c377c1d743569fc350651393f71a1a9e16d675b390a5c2f0bc97be10152fc3eefae028",
      "48fdb6f0ec250f238c9da3d


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] search_filters array: [{'label': 'is_fraud', 'value': '1'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp",
  "filters": [
    {
      "label": "is_fraud",
      "value": "1"
    }
  ],
  "limit": 10,
  "offset": 0
}


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "0461a05386c752640fb67219c1863c40b1774f930751aa180774b16c88d68d62": "d4c825400924747cda85960054ea116132632f39bf46877f3eb7dde2f206f008f748163f",
      "322db4f55171a681ce1b167427e3f53d2532c21960c2d19f479f9c86a8d8bc20": "28cd1508f036504e3a5ea03c41e967e127ef12f9de6c807a48cb92fd33b1cfe85ad1ff58",
      "3b9fbc9bf5f611b36303e1fa66fe9c10dc78104d499b7ff733ae7446cecda26c": "66c377c1d743569fc350651393f71a1a9e16d675b390a5c2f0bc97be10152fc3eefae028",
      "48fdb6f0ec250f238c9da3d


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


is_fraud:0 records found: 10
is_fraud:1 records found: 10

Sample is_fraud:0 record: {'data': {'amount': 69, 'country_DE': 0, 'country_FR': 0, 'country_NG': 0, 'country_TR': 0, 'country_UK': 1, 'country_US': 0, 'device_risk_score': 9, 'hour': 6, 'ip_risk_score': 15, 'is_fraud': 0, 'merchant_category_Clothing': 1, 'merchant_category_Electronics': 0, 'merchant_category_Food': 0, 'merchant_category_Grocery': 0, 'merchant_category_Travel': 0, 'transaction_id': 335, 'transaction_type_ATM': 0, 'transaction_type_Online': 0, 'transaction_type_POS': 0, 'transaction_type_QR': 1, 'user_id': 674}, 'id': 'LNXthWzmy69KmjhiFQAeGq', 'schema': 'https://localhost:8080/api/schemas/iEbXomadeyp3JcDfZMFLYp/', 'url': 'https://localhost:8080/api/records/LNXthWzmy69KmjhiFQAeGq/'}

Sample is_fraud:1 record: {'data': {'amount': 69, 'country_DE': 0, 'country_FR': 0, 'country_NG': 0, 'country_TR': 0, 'country_UK': 1, 'country_US': 0, 'device_risk_score': 9, 'hour': 6, 'ip_risk_score': 15, 'is_fraud': 0, 'merchant_

## Encrypted averages per batch (using transaction_id)
Compute integer-scaled means for each feature in batches using encrypted `avg` with a range filter on `transaction_id`. This demonstrates batch processing for the fraud dataset.

In [10]:
# Batch means (integer-scaled) over all rows, in batches using transaction_id
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
features = [
    "amount",
    "transaction_type_ATM",
    "transaction_type_Online",
    "transaction_type_POS",
    "transaction_type_QR",
    "merchant_category_Clothing",
    "merchant_category_Electronics",
    "merchant_category_Food",
    "merchant_category_Grocery",
    "merchant_category_Travel",
    "country_DE",
    "country_FR",
    "country_NG",
    "country_TR",
    "country_UK",
    "country_US",
    "hour",
]

batch_size = 10

# Reuse agg_value if defined; otherwise define here
def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# Find min and max transaction_id using encrypted aggregations
print("Finding transaction_id range...")
TOTAL_MAX = 100000  # a large upper bound for transaction_id

min_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter=f"transaction_id:min(0~{TOTAL_MAX})",
    decrypt=False,
    schema_id=SCHEMA_ID
)
min_id = int(agg_value(min_resp))

max_resp = client.aggregate(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    agg_filter=f"transaction_id:max(0~{TOTAL_MAX})",
    decrypt=False,
    schema_id=SCHEMA_ID
)
max_id = int(agg_value(max_resp))

print(f"Transaction ID range: {min_id} to {max_id}")
print(f"Processing in batches of {batch_size} transaction IDs...\n")

# Process in batches based on transaction_id ranges
start = min_id
batch_num = 0
while start <= max_id:
    end = min(start + batch_size - 1, max_id)
    batch_filter = f"transaction_id:{start}~{end}"
    batch_num += 1
    print(f"Batch {batch_num} (transaction_id: {start}-{end}):")
    for feature in features:
        try:
            resp = client.aggregate(
                organization=ORGANIZATION,
                dataset_slug=DATASET_SLUG,
                schema_slug=SCHEMA_SLUG,
                agg_filter=f"{feature}:avg(0~1000)",
                extra_filters=[batch_filter],
                decrypt=False,
                schema_id=SCHEMA_ID
            )
            mean_val = agg_value(resp)
            print(f"  mean {feature}: {mean_val:.3f}")
        except Exception as e:
            print(f"  Error with {feature}: {str(e)[:100]}")
    start += batch_size
    
    # Limit to first 3 batches for demo
    if batch_num >= 3:
        remaining_batches = (max_id - end) // batch_size + 1
        if remaining_batches > 0:
            print(f"\n... {remaining_batches} more batch(es) remaining (limited to 3 for demo)")
        break

print(f"\nBatch processing complete! Processed {batch_num} batch(es)")

Finding transaction_id range...
[DEBUG aggregate] Combined filters: ['transaction_id:min(0~100000)']
[DEBUG aggregate] extra_filters: None, agg_filter: transaction_id:min(0~100000)
[DEBUG query] search_filters array: [{'label': 'transaction_id', 'value': 'min(0~100000)'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp",
  "filters": [
    {
      "label": "transaction_id",
      "value": "min(0~100000)"
    }
  ],
  "limit": 1,
  "offset": 0
}


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "min",
      "value": 19
    },
    "id": "agg-1768104325869762",
    "schema": "https://localhost:8080/api/schemas/YpXTUQDoygjGNwrjSHmKLC/",
    "url": "https://localhost:8080/api/records/agg-1768104325869762/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'min', 'value': 19}, 'id': 'agg-1768104325869762', 'schema': 'https://localhost:8080/api/schemas/YpXTUQDoygjGNwrjSHmKLC/', 'url': 'https://localhost:8080/api/records/agg-1768104325869762/'}]}
[DEBUG aggregate] Combined filters: ['transaction_id:max(0~100000)']
[DEBUG aggregate] extra_filters: None, agg_filter: transaction_id:max(0~100000)
[DEBUG query] search_filters array: [{'label': 'transaction_id', 'value': 'max(0~100000)'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp",
  "filters": [
    {
      "label": "transaction_id",
      "value": "max(0~100000)"
    }
  ],
  "

/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "max",
      "value": 7253
    },
    "id": "agg-1768104326559248",
    "schema": "https://localhost:8080/api/schemas/YpXTUQDoygjGNwrjSHmKLC/",
    "url": "https://localhost:8080/api/records/agg-1768104326559248/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'max', 'value': 7253}, 'id': 'agg-1768104326559248', 'schema': 'https://localhost:8080/api/schemas/YpXTUQDoygjGNwrjSHmKLC/', 'url': 'https://localhost:8080/api/records/agg-1768104326559248/'}]}
Transaction ID range: 19 to 7253
Processing in batches of 10 transaction IDs...

Batch 1 (transaction_id: 19-28):
[DEBUG aggregate] Combined filters: ['transaction_id:19~28', 'amount:avg(0~1000)']
[DEBUG aggregate] extra_filters: ['transaction_id:19~28'], agg_filter: amount:avg(0~1000)
[DEBUG query] search_filters array: [{'label': 'transaction_id', 'value': '19~28'}, {'label': 'amount', 'value': 'avg(0~100

...

[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "avg",
      "value": null
    },
    "id": "agg-1768104337862092",
    "schema": "https://localhost:8080/api/schemas/hRNKWEuGviJezeLmsjC6oS/",
    "url": "https://localhost:8080/api/records/agg-1768104337862092/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'avg', 'value': None}, 'id': 'agg-1768104337862092', 'schema': 'https://localhost:8080/api/schemas/hRNKWEuGviJezeLmsjC6oS/', 'url': 'https://localhost:8080/api/records/agg-1768104337862092/'}]}
  mean country_UK: 0.000
[DEBUG aggregate] Combined filters: ['transaction_id:39~48', 'country_US:avg(0~1000)']
[DEBUG aggregate] extra_filters: ['transaction_id:39~48'], agg_filter: country_US:avg(0~1000)
[DEBUG query] search_filters array: [{'label': 'transaction_id', 'value': '39~48'}, {'label': 'country_US', 'value': 'avg(0~1000)'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp

/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "avg",
      "value": null
    },
    "id": "agg-1768104338145132",
    "schema": "https://localhost:8080/api/schemas/Z2k4BzU64N6BG758wUngtd/",
    "url": "https://localhost:8080/api/records/agg-1768104338145132/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'avg', 'value': None}, 'id': 'agg-1768104338145132', 'schema': 'https://localhost:8080/api/schemas/Z2k4BzU64N6BG758wUngtd/', 'url': 'https://localhost:8080/api/records/agg-1768104338145132/'}]}
  mean country_US: 0.000
[DEBUG aggregate] Combined filters: ['transaction_id:39~48', 'hour:avg(0~1000)']
[DEBUG aggregate] extra_filters: ['transaction_id:39~48'], agg_filter: hour:avg(0~1000)
[DEBUG query] search_filters array: [{'label': 'transaction_id', 'value': '39~48'}, {'label': 'hour', 'value': 'avg(0~1000)'}]
[DEBUG query] Final payload JSON: {
  "schema": "iEbXomadeyp3JcDfZMFLYp",
  "filters": [


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[DEBUG query] Response received: [
  {
    "data": {
      "aggregation_type": "avg",
      "value": null
    },
    "id": "agg-1768104338410853",
    "schema": "https://localhost:8080/api/schemas/Z7qv9SYz6DqfJbGFBuQ68w/",
    "url": "https://localhost:8080/api/records/agg-1768104338410853/"
  }
]
[DEBUG aggregate] Response received: {'success': True, 'records': [{'data': {'aggregation_type': 'avg', 'value': None}, 'id': 'agg-1768104338410853', 'schema': 'https://localhost:8080/api/schemas/Z7qv9SYz6DqfJbGFBuQ68w/', 'url': 'https://localhost:8080/api/records/agg-1768104338410853/'}]}
  mean hour: 0.000

... 721 more batch(es) remaining (limited to 3 for demo)

Batch processing complete! Processed 3 batch(es)


In [ ]:
# Example: Load data as a pandas DataFrame for exploration
client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)

# Check API health
health = client.health_check()
print(f"API Status: {health}")

# Load full dataset as DataFrame
# df = client.load_data(
#     organization=ORGANIZATION,
#     dataset_slug=DATASET_SLUG,
#     schema_slug=SCHEMA_SLUG,
#     limit=150
# )
df = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=150,
    decrypt=True,
    schema_id=SCHEMA_ID
)

print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataFrame info:")
print(df.info())


API Status: {'status': 'OK', 'message': 'BigQuery Schema Converter API is running (Test Mode)', 'timestamp': '2025-12-16T18:09:52.815Z'}

DataFrame shape: (150, 6)

Column names: ['dataset-order', 'petal-length', 'petal-width', 'sepal-length', 'sepal-width', 'species']

First few rows:
   dataset-order  petal-length  petal-width  sepal-length  sepal-width  \
0              1            14            2            51           35   
1              2            14            2            49           30   
2              3            13            2            47           32   
3              4            15            2            46           31   
4              5            14            3            50           36   

     species  
0  I. setosa  
1  I. setosa  
2  I. setosa  
3  I. setosa  
4  I. setosa  

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------   

## Summary

This notebook demonstrates:

1. ✅ Loading data from Blind Insight instead of sklearn datasets
2. ✅ Using the data with scikit-learn for machine learning
3. ✅ Training a logistic regression classifier
4. ✅ Evaluating model accuracy

The key difference from the standard scikit-learn example is that instead of:
```python
iris = datasets.load_iris()
```

We use:
```python
X, y = load_iris_from_blind(organization, dataset_slug, schema_slug)
```

This allows data scientists to work with encrypted, privacy-preserving data stored in Blind Insight while using standard ML libraries and workflows.
